In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
(xf_train, yf_train), (xf_test, yf_test) = keras.datasets.fashion_mnist.load_data()


In [3]:
x_train, x_test = x_train / 255.0, x_test / 255.0
xf_train, xf_test = xf_train / 255.0, xf_test / 255.0

In [4]:
def process_labels(y):
    y_new = np.where(y == 3, 0, y)
    return np.where(y_new > 8, 8, y_new)

y_train_ff = process_labels(yf_train)
y_test_ff = process_labels(yf_test)


In [5]:
def build_tf_model(output_units):
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(name="shared_flatten"),
        keras.layers.Dense(128, activation='relu', name="shared_dense_1"),
        keras.layers.Dropout(0.3, name="shared_dropout"),
        keras.layers.BatchNormalization(name="shared_bn"),
        keras.layers.Dense(64, activation='relu', name="shared_dense_2"),
        keras.layers.Dense(output_units, activation='softmax', name="layer_output")
    ])

print("--- ЭТАП 1: Обучение базовых моделей на MNIST ---")


--- ЭТАП 1: Обучение базовых моделей на MNIST ---


In [21]:
model_10 = build_tf_model(10)
model_10.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_10.fit(x_train, y_train, epochs=2, batch_size=64, verbose=0)
_, acc_tf_10 = model_10.evaluate(x_test, y_test, verbose=0)
print(f"TF M Accuracy: {acc_tf_10:.4f}")

TF M Accuracy: 0.9677


In [22]:
tf_weights_path = "shared_tf_weights.weights.h5"
model_10.save_weights(tf_weights_path)
print(f"Веса TF сохранены в {tf_weights_path}")

Веса TF сохранены в shared_tf_weights.weights.h5


In [23]:
class PyTorchNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Flatten(), nn.Linear(28*28, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.BatchNorm1d(128),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.classifier = nn.Linear(64, num_classes)
    def forward(self, x): return self.classifier(self.features(x))

In [24]:
pt_model_10 = PyTorchNet(10)
crit = nn.CrossEntropyLoss() # Исправлено: определение crit
opt10 = torch.optim.Adam(pt_model_10.parameters(), lr=0.001)

In [25]:
def get_pt_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in loader:
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return correct / total

In [26]:
pt_loader_mnist = DataLoader(TensorDataset(torch.tensor(x_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), batch_size=64, shuffle=True)
pt_test_loader_mnist = DataLoader(TensorDataset(torch.tensor(x_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), batch_size=64)

In [27]:
pt_model_10.train()
for i, (d, t) in enumerate(pt_loader_mnist):
    opt10.zero_grad()
    loss = crit(pt_model_10(d), t)
    loss.backward()
    opt10.step()
    if i == 200: break

# Оценка
acc_pt_10 = get_pt_accuracy(pt_model_10, pt_test_loader_mnist)
print(f"PyTorch M Accuracy: {acc_pt_10:.4f}")

PyTorch M Accuracy: 0.9286


In [28]:
pt_weights_path = "shared_pt_weights.pth"
torch.save(pt_model_10.state_dict(), pt_weights_path)
print(f"Веса PyTorch сохранены в {pt_weights_path}")

Веса PyTorch сохранены в shared_pt_weights.pth


In [29]:
model_9 = build_tf_model(9)
# Загружаем веса только для общих слоев (без предупреждений)
for layer in model_9.layers:
    if layer.name != "layer_output":
        layer.set_weights(model_10.get_layer(layer.name).get_weights())

In [30]:
model_9.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_9.fit(xf_train, y_train_ff, epochs=2, batch_size=64, verbose=0)
_, acc_tf_9 = model_9.evaluate(xf_test, y_test_ff, verbose=0)
print(f"TF F Accuracy: {acc_tf_9:.4f}")

TF F Accuracy: 0.8498


In [31]:
pt_model_9 = PyTorchNet(9)
loaded_dict = torch.load(pt_weights_path)
current_dict = pt_model_9.state_dict()

filtered_dict = {k: v for k, v in loaded_dict.items() if k in current_dict and v.size() == current_dict[k].size()}
current_dict.update(filtered_dict)
pt_model_9.load_state_dict(current_dict)

<All keys matched successfully>

In [32]:
crit = nn.CrossEntropyLoss()
opt9 = torch.optim.Adam(pt_model_9.parameters(), lr=0.001)
pt_loader_fashion = DataLoader(TensorDataset(torch.tensor(xf_train, dtype=torch.float32), torch.tensor(y_train_ff, dtype=torch.long)), batch_size=64, shuffle=True)

In [33]:
pt_model_9.train()
for i, (d, t) in enumerate(pt_loader_fashion):
    opt9.zero_grad(); crit(pt_model_9(d), t).backward(); opt9.step()
    if i == 200: break

In [34]:
pt_model_9.eval()
correct, total = 0, 0
with torch.no_grad():
    for d, t in DataLoader(TensorDataset(torch.tensor(xf_test, dtype=torch.float32), torch.tensor(y_test_ff, dtype=torch.long)), batch_size=64):
        _, p = torch.max(pt_model_9(d), 1); total += t.size(0); correct += (p == t).sum().item()
acc_pt_9 = correct / total
print(f"PyTorch F Accuracy: {acc_pt_9:.4f}")

PyTorch F Accuracy: 0.8139


In [37]:
results_data = {
    'Framework': ['TensorFlow', 'TensorFlow', 'PyTorch', 'PyTorch'],
    'Dataset': ['MNIST', 'Fashion ', 'MNIST', 'Fashion'],
    'Classes': [10, 9, 10, 9],
    'Accuracy': [acc_tf_10, acc_tf_9, acc_pt_10, acc_pt_9] # Здесь теперь переменная вместо текста
}

df = pd.DataFrame(results_data)
df.to_csv('model_results.csv', index=False)
print("\n[УСПЕХ] Результаты сохранены в файл 'model_results.csv'")
print(df)


[УСПЕХ] Результаты сохранены в файл 'model_results.csv'
    Framework   Dataset  Classes  Accuracy
0  TensorFlow     MNIST       10    0.9677
1  TensorFlow  Fashion         9    0.8498
2     PyTorch     MNIST       10    0.9286
3     PyTorch   Fashion        9    0.8139
